# Phase 6 — Cross-Method Benchmarks & Analysis

**Goal:** assemble a single picture from all method-specific notebooks. This is the notebook that produces the headline tables for the thesis writeup.

**Inputs (Kaggle datasets):**
- `sqat-baseline`     — FP32 numbers from notebook 01
- `sqat-ptq`          — PTQ INT8 / INT4
- `sqat-standard-qat` — Standard QAT INT8 / INT4
- `sqat-scheduled-qat` — Scheduled QAT linear / cosine / step
- `sqat-lora-qat`     — LoRA-QAT INT8 / INT4
- `sqat-gguf` *(optional)* — GGUF file sizes

**Outputs:**
- `results/tables/primary_results.csv`
- `results/tables/schedule_comparison.csv`
- `results/plots/*.png`

This notebook is CPU-only — no GPU required.

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q -e ".[viz]"

In [ ]:
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TABLES_DIR = Path(WORKING_DIR) / "results" / "tables"
PLOTS_DIR  = Path(WORKING_DIR) / "results" / "plots"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load all results

Each phase notebook writes a per-experiment `training_results.json` plus a method-level summary JSON. We glob both, merge into one DataFrame, then add derived columns.

In [ ]:
DATASET_ROOTS = [
    Path("/kaggle/input/sqat-baseline"),
    Path("/kaggle/input/sqat-ptq"),
    Path("/kaggle/input/sqat-standard-qat"),
    Path("/kaggle/input/sqat-scheduled-qat"),
    Path("/kaggle/input/sqat-lora-qat"),
    Path(WORKING_DIR),                # in case you re-ran phases here
]

# Approximate model sizes (GB) per bit-width. Real GGUF sizes can replace these
# if you mount sqat-gguf — see Section 6.
APPROX_SIZES = {32: 6.5, 16: 3.4, 8: 1.7, 4: 0.85}

def find_results():
    rows = []
    for root in DATASET_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("training_results.json"):
            with path.open() as f:
                d = json.load(f)
            d["_source"] = str(path)
            rows.append(d)
    return rows

def load_baseline():
    for root in DATASET_ROOTS:
        p = root / "results" / "baseline" / "baseline_results.json"
        if p.exists():
            with p.open() as f:
                return json.load(f)
    return None

fp32 = load_baseline()
raw = find_results()
print(f"Loaded {len(raw)} experiment result files.")
if fp32:
    print(f"FP32 perplexity: {fp32.get('perplexity'):.4f}")

In [ ]:
def schedule_of(name: str) -> str:
    # scheduled_qat_cosine_int4 -> cosine ; standard_qat_int4 -> '' ; ptq_int4 -> ''
    parts = name.split("_")
    if parts[:2] == ["scheduled", "qat"] and len(parts) >= 4:
        return parts[2]
    return ""

rows = []
fp32_ppl = fp32["perplexity"] if fp32 else None

for d in raw:
    name = d.get("experiment_name", Path(d["_source"]).parent.name)
    bits = d.get("target_bits")
    method = d.get("method", name.split("_int")[0])
    ppl = d.get("perplexity")
    rows.append({
        "experiment_name": name,
        "method":          method,
        "schedule":        schedule_of(name),
        "bits":            bits,
        "perplexity":      ppl,
        "kl_divergence":   d.get("kl_divergence"),
        "final_loss":      d.get("final_loss"),
        "training_time_s": d.get("training_time_seconds"),
        "size_gb":         APPROX_SIZES.get(bits),
        "ppl_delta_pct":   ((ppl / fp32_ppl) - 1) * 100 if (ppl and fp32_ppl) else None,
    })

df = pd.DataFrame(rows).sort_values(["method", "bits", "schedule"]).reset_index(drop=True)
df.round(4)

## 3. Primary results table

The headline table from `SKILL.md`. Lower PPL and lower KLD = better.

In [ ]:
primary_cols = ["method", "schedule", "bits", "perplexity", "ppl_delta_pct",
                "kl_divergence", "size_gb", "training_time_s"]
primary = df[primary_cols].copy()

# Prepend the FP32 baseline row.
if fp32:
    primary = pd.concat([
        pd.DataFrame([{
            "method": "baseline", "schedule": "", "bits": 32,
            "perplexity": fp32["perplexity"], "ppl_delta_pct": 0.0,
            "kl_divergence": 0.0, "size_gb": 6.5, "training_time_s": 0,
        }]),
        primary,
    ], ignore_index=True)

primary = primary.round({"perplexity": 4, "ppl_delta_pct": 2,
                          "kl_divergence": 6, "size_gb": 2, "training_time_s": 0})
primary.to_csv(TABLES_DIR / "primary_results.csv", index=False)
primary

## 4. Schedule comparison (INT4)

The thesis-specific table. Which schedule wins, and by how much?

In [ ]:
sched_df = df[(df["method"] == "scheduled_qat") & (df["bits"] == 4)][
    ["schedule", "perplexity", "ppl_delta_pct", "kl_divergence", "final_loss", "training_time_s"]
].sort_values("kl_divergence").reset_index(drop=True)
sched_df = sched_df.round({"perplexity": 4, "ppl_delta_pct": 2,
                            "kl_divergence": 6, "final_loss": 4, "training_time_s": 0})
sched_df.to_csv(TABLES_DIR / "schedule_comparison.csv", index=False)
sched_df

## 5. Plots

In [ ]:
# Perplexity grouped by method × bits.
plot_df = df.dropna(subset=["perplexity"]).copy()
if not plot_df.empty:
    pivot = plot_df.pivot_table(
        index="bits", columns="method", values="perplexity", aggfunc="min",
    ).sort_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    pivot.plot(kind="bar", ax=ax)
    if fp32:
        ax.axhline(fp32["perplexity"], linestyle="--", color="black",
                   label=f"FP32 ({fp32['perplexity']:.2f})")
    ax.set_ylabel("Perplexity (lower = better)")
    ax.set_title("WikiText-103 perplexity by method and bits")
    ax.legend(title="method")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "ppl_vs_bits.png", dpi=150)
    plt.show()

In [ ]:
# KL divergence heatmap.
kld_df = df.dropna(subset=["kl_divergence"])
if not kld_df.empty:
    pivot = kld_df.pivot_table(
        index="method", columns="bits", values="kl_divergence", aggfunc="min",
    )
    fig, ax = plt.subplots(figsize=(7, max(3, len(pivot) * 0.5)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
    ax.set_xlabel("bits"); ax.set_ylabel("method")
    ax.set_title("KL divergence vs FP32 (lower = better)")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.4f}", ha="center", va="center",
                        color="white" if v > pivot.values[~np.isnan(pivot.values)].mean() else "black")
    fig.colorbar(im, ax=ax, label="KLD")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "kld_heatmap.png", dpi=150)
    plt.show()

In [ ]:
# Schedule comparison bars (INT4 only).
if not sched_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(sched_df["schedule"], sched_df["perplexity"],
                color=["tab:blue", "tab:orange", "tab:green"][: len(sched_df)])
    if fp32:
        axes[0].axhline(fp32["perplexity"], linestyle="--", color="black")
    axes[0].set_ylabel("Perplexity")
    axes[0].set_title("INT4 perplexity by schedule")
    axes[1].bar(sched_df["schedule"], sched_df["kl_divergence"],
                color=["tab:blue", "tab:orange", "tab:green"][: len(sched_df)])
    axes[1].set_ylabel("KL divergence vs FP32")
    axes[1].set_title("INT4 KL divergence by schedule")
    for ax in axes:
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "schedule_comparison.png", dpi=150)
    plt.show()

In [ ]:
# Quality-vs-size scatter — useful for identifying Pareto-frontier methods.
scatter = df.dropna(subset=["perplexity", "size_gb"]).copy()
if not scatter.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    for method, g in scatter.groupby("method"):
        ax.scatter(g["size_gb"], g["perplexity"], label=method, s=80, alpha=0.8)
        for _, row in g.iterrows():
            label = f"{row['schedule']} INT{row['bits']}" if row["schedule"] else f"INT{row['bits']}"
            ax.annotate(label, (row["size_gb"], row["perplexity"]),
                        fontsize=8, alpha=0.7, xytext=(4, 4), textcoords="offset points")
    if fp32:
        ax.scatter([6.5], [fp32["perplexity"]], marker="*", s=200, color="black", label="FP32")
    ax.set_xlabel("Approx. model size (GB)")
    ax.set_ylabel("Perplexity (lower = better)")
    ax.set_title("Quality vs size — Pareto view")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "quality_vs_size.png", dpi=150)
    plt.show()

## 6. (Optional) Real GGUF sizes

If you mount the `sqat-gguf` dataset, this cell replaces the approximate sizes with measured `.gguf` file sizes.

In [ ]:
GGUF_ROOT = Path("/kaggle/input/sqat-gguf")
if GGUF_ROOT.exists():
    rows = []
    for p in GGUF_ROOT.rglob("*.gguf"):
        rows.append({"name": p.name, "size_gb": round(p.stat().st_size / 1e9, 3)})
    gguf_df = pd.DataFrame(rows).sort_values("size_gb")
    print(gguf_df.to_string(index=False))
    gguf_df.to_csv(TABLES_DIR / "gguf_sizes.csv", index=False)
else:
    print("sqat-gguf dataset not mounted — skipping. Approx sizes used in tables above.")

## 7. Conclusions cell

Auto-summarises the headline findings: best INT4 method, best schedule, and Scheduled-vs-Standard QAT improvement.

In [ ]:
lines = []

int4 = df[df["bits"] == 4].dropna(subset=["kl_divergence"])
if not int4.empty:
    best = int4.sort_values("kl_divergence").iloc[0]
    lines.append(f"Best INT4 by KLD: {best['experiment_name']}  "
                 f"(KLD={best['kl_divergence']:.6f}, PPL={best['perplexity']:.4f})")

if not sched_df.empty:
    winner = sched_df.iloc[0]
    lines.append(f"Best schedule (INT4): {winner['schedule']}  "
                 f"(KLD={winner['kl_divergence']:.6f})")

std_int4 = df[(df["method"] == "standard_qat") & (df["bits"] == 4)]
sch_int4 = df[(df["method"] == "scheduled_qat") & (df["bits"] == 4)]
if not std_int4.empty and not sch_int4.empty:
    std_kld = std_int4["kl_divergence"].min()
    sch_kld = sch_int4["kl_divergence"].min()
    if pd.notna(std_kld) and pd.notna(sch_kld) and std_kld > 0:
        improvement = (std_kld - sch_kld) / std_kld * 100
        lines.append(f"Scheduled vs Standard QAT @ INT4: "
                     f"KLD {std_kld:.6f} -> {sch_kld:.6f}  ({improvement:+.1f}%)")

for l in lines:
    print(l)

with (TABLES_DIR / "conclusions.txt").open("w") as f:
    f.write("\n".join(lines) + "\n")

In [ ]:
!ls -lh {TABLES_DIR} {PLOTS_DIR}